# Versuch BKL: Blutkreislauf

| | |
|---|---|
| **Name** | *Vorname Nachname* |
| **Matrikelnummer** | *123456* |
| **Gruppe** | *…* |
| **Betreuer:in** | *…* |
| **Versuchstag** | *TT.MM.JJJJ* |

**Vorübergehend:** Im Praktikumsprofil fehlen derzeit noch Pakete, die dieses Notebook braucht. Führen Sie die folgende Zelle deshalb zu Beginn einmal aus, sie installiert die Pakete. Meldet ein späterer Import trotzdem ein fehlendes Paket, starten Sie den Kernel neu (**Kernel** → **Restart Kernel**) und führen Sie die Zellen erneut aus.

In [ ]:
%pip install --quiet numpy scipy matplotlib uncertainties odrpack==0.6.1

In diesem Notebook werden die Messdaten zum Versuch BKL ausgewertet (Anleitung Kapitel 4, Auswertung in Abschnitt 4.4).

Es werden verwendet:

- `numpy` für Rechnungen mit Messwerten
- `matplotlib.pyplot` für Diagramme
- `odr_fit` aus `odrpack` für die linearen Fits mit Unsicherheiten in beide Richtungen

Die Auswertung besteht aus fünf Teilen:

1. Funktion des Windkessels (Abschnitt 4.4.1)
2. Leitwert von Rohr 2 und seine Unsicherheit (Aufgabe 1)
3. Leitwerte und Strömungswiderstände aller Messreihen (Aufgaben 2 und 3)
4. Vergleich der Rohre (Aufgaben 4 bis 6)
5. Turbulenzrohr (Aufgabe 7)

Notation wie im Versuch MEDA: $\bar x$ Mittelwert, $s_x$ Standardabweichung, $u_x$ Messunsicherheit. Im Bericht werden Ergebnisse in der Form $(\bar x \pm u_x)\,\mathrm{Einheit}$ angegeben. $u_x$ wird auf eine signifikante Stelle aufgerundet (auf zwei, wenn die erste Stelle eine 1 oder 2 ist), der Bestwert auf dieselbe Dezimalstelle gerundet.

Die Anleitung schreibt die Unsicherheit des Leitwerts als $\Delta G$, hier heißt sie $u_G$. $\Delta p$, $\Delta t$ und $\Delta x$ sind dagegen Messgrößen (Druckdifferenz, Sinkzeit, Sinkstrecke).

Die verwendeten Python-Werkzeuge (Arrays, `plt.errorbar()`, `odr_fit`, Fehlerfortpflanzung) kennen Sie aus der Einführung PYT.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from odrpack import odr_fit

## 1. Funktion des Windkessels

Im Versuch wurden die minimale und die maximale Druckdifferenz $\Delta p_\mathrm{min}$ und $\Delta p_\mathrm{max}$ während eines Pumpzyklus abgelesen, einmal ohne und einmal mit Windkessel (Abschnitt 4.3.1). Dieser Teil wird nicht gerechnet, sondern in eigenen Worten beschrieben (Abschnitt 4.4.1):

- Wie beeinflusst der Windkessel die auftretenden Druckschwankungen und das Strömungsverhalten?
- Warum ist diese Funktion für den Blutkreislauf von Bedeutung?
- Wie wird die Funktion des Windkessels im Blutkreislauf realisiert?

Beziehen Sie sich dabei auf Ihre Messwerte.

## 2. Leitwert von Rohr 2

In diesem Teil wird aus den fünffach wiederholten Messungen an Rohr 2 die Volumenstromstärke $j$ berechnet und gegen die Druckdifferenz $\Delta p$ aufgetragen. Die Steigung ist der Strömungsleitwert $G$ (Gl. 4.2):

$$
j = G \cdot \Delta p
$$

Die Volumenstromstärke folgt aus Sinkstrecke $\Delta x$ und Sinkzeit $\Delta t$ des Kolbens (Gl. 4.5):

$$
j = \frac{A \cdot \Delta x}{\Delta t}
$$

Zuerst werden die Messwerte definiert. Danach werden $j$ und der Leitwert berechnet, dessen Unsicherheit $u_G$ nach Aufgabe 1 aus den Geraden minimaler und maximaler Steigung folgt.

### Messwerte und Konstanten definieren

Hier werden die Sinkstrecken $\Delta x$ und die je fünfmal gemessenen Druckdifferenzen $\Delta p$ und Sinkzeiten $\Delta t$ definiert (Tabelle 4.2). Die fünf Werte eines Massestücks bilden eine Zeile eines zweidimensionalen Arrays: `np.array([[...], [...], [...], [...]])`.

Gerechnet wird in den Einheiten der Anleitung: $\Delta x$ in $\mathrm{mm}$, $\Delta p$ in $\mathrm{hPa}$, $\Delta t$ in $\mathrm{s}$, $j$ in $\mathrm{mm^3/s}$.

Vorgabe aus der Anleitung (Gl. 4.5): $A = 5027\,\mathrm{mm^2}$. Massen und Strecken werden ohne Unsicherheit verwendet (Abschnitt 4.3.2).

In [ ]:
# Ihr Code hier


### Mittelwerte und Unsicherheiten berechnen

Die Messaufgabe in Abschnitt 4.3.2 verlangt für jedes Massestück den Mittelwert und die Messunsicherheit von $\Delta p$ und $\Delta t$. Die Mittelwerte werden mit `np.mean()` berechnet, die Unsicherheit des Mittelwerts über

$$
u_{\bar x} =
\frac{s_x}{\sqrt{N}}
$$

$N$ ist die Anzahl der Messwerte, $s_x$ die Standardabweichung mit dem Nenner $N-1$:

$$
s_x =
\sqrt{
\frac{1}{N-1}
\sum_{i=1}^{N}
(x_i-\bar x)^2
}
$$

Die Standardabweichung wird mit `np.std(x, ddof=1)` berechnet. Ohne `ddof=1` verwendet `np.std()` den Nenner $N$.

Bei einem zweidimensionalen Array mittelt `np.mean(dp_2, axis=1)` jede Zeile für sich und liefert so die vier Mittelwerte auf einmal, `np.std(dp_2, axis=1, ddof=1)` entsprechend. `len(dp_2[0])` ist die Anzahl der Werte in der ersten Zeile, also die Zahl der Wiederholungen.

In [ ]:
# Ihr Code hier


### Volumenstromstärke berechnen

Mit den Mittelwerten der Sinkzeiten wird für jedes Massestück die Volumenstromstärke nach Gl. 4.5 berechnet.

Die Unsicherheit von $j$ wird mit Gaußscher Fehlerfortpflanzung bestimmt. Überlegen Sie dazu, mit welcher Messunsicherheit $j$ angegeben werden muss (Abschnitt 4.3.2). Massen und Strecken werden ohne Unsicherheit verwendet, die Kolbenfläche $A$ ist vorgegeben. Leiten Sie $j$ nach den Größen ab, die eine Unsicherheit tragen.

**Hinweis:** Die Fehlerfortpflanzung kann auch mit dem Paket `uncertainties` berechnet werden. Dazu werden die Messgrößen zusammen mit ihren Unsicherheiten als Arrays angelegt, zum Beispiel `dt_u = unp.uarray(mean_dt_2, u_dt_2)` (nach `import uncertainties.unumpy as unp`). Mit diesen Arrays wird wie gewohnt gerechnet, Funktionen wie Wurzel werden dabei als `unp.sqrt()` geschrieben. Bestwert und Unsicherheit des Ergebnisses liefern `unp.nominal_values(j)` und `unp.std_devs(j)`. Achten Sie auf die richtige Angabe signifikanter Stellen. `uncertainties` rundet Unsicherheiten kaufmännisch statt aufzurunden.

In [ ]:
# Ihr Code hier


### Diagramm der Messwerte erstellen

Vor dem Fit werden die Messwerte dargestellt: $j$ gegen $\Delta p$ mit Fehlerbalken $u_{\Delta p}$ und $u_j$. Prüfen Sie am Verlauf, ob der lineare Zusammenhang nach Gl. 4.2 vorliegt.

Zur Erinnerung: `plt.errorbar(x, y, xerr=u_x, yerr=u_y, marker="o", linestyle="", label="Messwerte")` zeichnet die Messwerte mit Fehlerbalken.

Achsen werden in der Form „Größe in Einheit“ beschriftet.

In [ ]:
# Ihr Code hier


### Leitwert durch linearen Fit bestimmen

Die Ausgleichsgerade hat die Form

$$
j = m \cdot \Delta p + b
$$

Aus dem Vergleich mit Gl. 4.2 folgt $G = m$. Der Fit liefert außerdem den Achsenabschnitt $b$.

Weil beide Größen Unsicherheiten tragen, wird der Fit mit `odr_fit` durchgeführt. Er gewichtet die Messwerte mit ihren Unsicherheiten in beide Richtungen, genau wie die Methode minimaler und maximaler Steigung aus Abschnitt 1.3.2 die Fehlerbalken in $x$- und $y$-Richtung verwendet. Aufgabe 1 verlangt diese Methode zusätzlich, sie folgt im nächsten Schritt.

Zur Erinnerung: ODR berücksichtigt Unsicherheiten in $x$- und $y$-Richtung. Der Fit läuft in vier Schritten.

**1. Modellfunktion definieren.** Die Parameter stehen gemeinsam im Array `beta`. Die Funktion bekommt zuerst `x` und danach `beta`:

```python
def linear_odr(x, beta):
    return beta[0] * x + beta[1]
```

**2. Gewichte berechnen.** `odr_fit` erwartet die Unsicherheiten nicht direkt, sondern als Gewichte. Das Gewicht eines Messwerts ist eins geteilt durch das Quadrat seiner Unsicherheit:

$$
w_x = \frac{1}{u_x^2}
\qquad
w_y = \frac{1}{u_y^2}
$$

**3. Fit durchführen.** Übergeben werden die Modellfunktion, die Messwerte, die Startwerte `beta0` und die Gewichte:

```python
output = odr_fit(linear_odr, x, y, beta0=[1, 0], weight_x=1/u_x**2, weight_y=1/u_y**2)
```

**4. Ergebnisse auslesen.** Die Parameter stehen in `output.beta`. Ihre Unsicherheiten stehen wie bei `curve_fit` auf der Diagonalen der Kovarianzmatrix, hier heißt sie `output.cov_beta`:

```python
u_m = np.sqrt(output.cov_beta[0, 0])   # Steigung
u_b = np.sqrt(output.cov_beta[1, 1])   # Achsenabschnitt
```

**Wichtig:** `beta0` sind die Startwerte. `odr_fit` arbeitet mit relativen Schrittweiten und braucht sie in der richtigen Größenordnung, bei sehr kleinen Zahlen darf kein Startwert genau null sein.

$G$ wird in $\mathrm{mm^3/(hPa\,s)}$ ausgegeben, $b$ in $\mathrm{mm^3/s}$.

In [ ]:
# Ihr Code hier


### Aufgabe 1: Unsicherheit des Leitwerts nach der Methode minimaler und maximaler Steigung bestimmen

Aufgabe 1 verlangt, die Unsicherheit $u_G$ graphisch aus den Geraden minimaler und maximaler Steigung zu ermitteln. Dazu werden zwei Grenzgeraden durch den Messpunkt mit der kleinsten und den mit der größten Druckdifferenz gelegt (Methode aus Abschnitt 1.3.2 der Anleitung, Indizes mit `np.argmin()` und `np.argmax()`). Mit $x = \Delta p$ und $y = j$:

- minimale Steigung durch $(x_\mathrm{min} - u_x,\ y + u_y)$ und $(x_\mathrm{max} + u_x,\ y - u_y)$,
- maximale Steigung durch $(x_\mathrm{min} + u_x,\ y - u_y)$ und $(x_\mathrm{max} - u_x,\ y + u_y)$.

Die Steigung einer Geraden durch zwei Punkte ist der Quotient aus $y$- und $x$-Differenz. Die größere Abweichung zur Steigung des Fits ist die Unsicherheit:

$$
u_G = \max\left( |G - G_\mathrm{max}|,\ |G - G_\mathrm{min}| \right)
$$

Für das Diagramm im nächsten Schritt wird auch der Achsenabschnitt jeder Grenzgeraden berechnet: $b = y_1 - m\,x_1$ mit dem ersten Punkt $(x_1, y_1)$.

In [ ]:
# Ihr Code hier


### Aufgabe 1: Diagramm mit Fitgerade und Grenzgeraden erstellen

Die Fitgerade und die beiden Grenzgeraden werden in das Diagramm der Messwerte eingezeichnet.

Zur Erinnerung: `plt.plot(x, linear_odr(x, output.beta), label="Fit")` zeichnet die Fitgerade an den $x$-Werten der Messung. Mit `linestyle="--"` werden die Grenzgeraden gestrichelt gezeichnet.

In [ ]:
# Ihr Code hier


## 3. Leitwerte und Strömungswiderstände

In diesem Teil werden die Messungen an Rohr 3, an Rohr 1 und an der Parallelschaltung von Rohr 2 und Rohr 3 ausgewertet. Alle vier Messreihen werden gemeinsam dargestellt (Aufgabe 2). Aus den Steigungen folgen die Leitwerte, aus ihren Kehrwerten die Strömungswiderstände, die mit den theoretischen Werten nach Gl. 4.3 verglichen werden (Aufgabe 3).

### Messwerte definieren

Hier werden für Rohr 3, Rohr 1 und die Parallelschaltung die Sinkstrecken $\Delta x$, die Druckdifferenzen $\Delta p$ und die Sinkzeiten $\Delta t$ definiert (Tabellen 4.3 bis 4.5). Jede Messung wurde einmal durchgeführt.

In [ ]:
# Ihr Code hier


### Volumenstromstärken berechnen

Berechnen Sie für jede Messreihe $j$ nach Gl. 4.5 und $u_j$ wie in Teil 2. Nach Abschnitt 4.3.2 wird die an Rohr 2 bestimmte Messunsicherheit für alle weiteren Messungen als Schätzwert übernommen.

In [ ]:
# Ihr Code hier


### Aufgabe 2: Diagramm aller Messreihen erstellen

Es wird $j$ gegen $\Delta p$ für Rohr 2, Rohr 3, Rohr 1 und die Parallelschaltung in einem gemeinsamen Diagramm dargestellt, jede Messreihe in einer eigenen Farbe und mit Legende. Überlegen Sie vorher, wie das Diagramm dimensioniert sein muss, damit sich alle Messreihen vergleichen lassen (Aufgabe 2). Die Achsengrenzen werden mit `plt.xlim()` und `plt.ylim()` festgelegt.

Aufgabe 2 fragt außerdem: Welchen generellen Zusammenhang zwischen Volumenstromstärke und Druckdifferenz vermuten Sie aufgrund des Diagramms? Wie verhält sich die Strömung bei einem Druck von $0\,\mathrm{Pa}$?

In [ ]:
# Ihr Code hier


### Aufgabe 3: Leitwerte durch lineare Fits bestimmen

Bestimmen Sie die Leitwerte von Rohr 3, Rohr 1 und der Parallelschaltung wie in Teil 2 aus der Steigung eines Fits mit `odr_fit`. Berücksichtigen Sie dabei die Frage aus Aufgabe 3: Geht die Gerade durch den Nullpunkt?

Zur Erinnerung (ODR), Vorgehen wie beim ersten Fit:

- Modellfunktion `linear_odr(x, beta)`: zuerst `x`, danach `beta`
- Gewichte `weight_x=1/u_x**2` und `weight_y=1/u_y**2`
- Startwerte `beta0` in der richtigen Größenordnung
- Unsicherheiten der Parameter aus `np.sqrt(output.cov_beta[0, 0])` und `np.sqrt(output.cov_beta[1, 1])`

Für die Unsicherheiten der Leitwerte gilt Aufgabe 1: Die an Rohr 2 graphisch ermittelte Unsicherheit $u_G$ wird für alle Ergebnisse verwendet.

In [ ]:
# Ihr Code hier


### Aufgabe 3: Diagramm mit den Fitgeraden erstellen

Die Fitgeraden aller vier Messreihen werden in das Diagramm aus Aufgabe 2 eingezeichnet. Mit `color="C0"` bis `color="C3"` erhalten Messwerte und Fitgerade einer Messreihe dieselbe Farbe.

In [ ]:
# Ihr Code hier


### Aufgabe 3: Strömungswiderstände berechnen

Der Strömungswiderstand ist der Kehrwert des Leitwerts:

$$
R = \frac{1}{G}
$$

Die Unsicherheit folgt aus Gaußscher Fehlerfortpflanzung über $G$. Achten Sie auf die Einheiten (Aufgabe 3): Gl. 4.2 gibt $R$ in $\mathrm{Pa\,s/m^3}$ an.

In [ ]:
# Ihr Code hier


### Aufgabe 3: Theoretische Strömungswiderstände berechnen und vergleichen

Theoretisch gilt bei laminarer Strömung (Gl. 4.3):

$$
R = \frac{8\,\eta \cdot l}{\pi \cdot r^4}
$$

Radien und Längen stehen in Aufgabe 3: $r_1 = 3\,\mathrm{mm}$, $r_2 = r_3 = 6\,\mathrm{mm}$, $l_1 = l_2 = 388\,\mathrm{mm}$, $l_3 = 831\,\mathrm{mm}$. Für $\eta$ wird der Wert verwendet, den Sie bei der Raumtemperatur aus Abb. 4.9 abgelesen haben (Abschnitt 4.3.4).

Vergleichen Sie die theoretischen Werte mit den gemessenen Werten der Rohre 1, 2 und 3 (Aufgabe 3 verlangt dafür im Bericht eine Tabelle). Für den Vergleich eignen sich die relative Abweichung in Prozent und die Abweichung in Vielfachen der Unsicherheit.

In [ ]:
# Ihr Code hier


## 4. Vergleich der Rohre

In diesem Teil werden die Ergebnisse der Rohre untereinander verglichen (Aufgaben 4 bis 6). Gerechnet wird mit den Leitwerten und Strömungswiderständen aus Teil 2 und Teil 3.

### Aufgabe 4: Rohr 3 mit Rohr 2 vergleichen

Vergleichen Sie die Ergebnisse von Rohr 3 mit denen von Rohr 2. Rohr 2 hat denselben Durchmesser, ist aber nur etwa halb so lang.

Aufgabe 4 fragt außerdem: Entspricht Ihre Beobachtung Ihren Erwartungen? Warum kann aus den Messungen an Rohr 2 und Rohr 3 auch auf den Strömungswiderstand einer Serienschaltung geschlossen werden? Wie verhält sich das bei einer Reihenschaltung elektrischer Widerstände?

In [ ]:
# Ihr Code hier


### Aufgabe 5: Rohr 1 mit Rohr 2 vergleichen

Rohr 1 hat dieselbe Länge wie Rohr 2, aber etwa den halben Durchmesser. Aufgabe 5 fragt, um welchen Faktor sich die Strömungswiderstände unterscheiden, also nach dem Verhältnis

$$
\frac{R_1}{R_2}
$$

Berechnen Sie es zusammen mit seiner Unsicherheit (Gaußsche Fehlerfortpflanzung).

Aufgabe 5 fragt außerdem: Entspricht Ihre Beobachtung Ihren Erwartungen? Wie ändert sich der elektrische Widerstand eines Leiters, wenn sein Durchmesser verdoppelt wird?

In [ ]:
# Ihr Code hier


### Aufgabe 6: Parallelschaltung mit den Einzelrohren vergleichen

Vergleichen Sie den Leitwert der Parallelschaltung mit den Leitwerten von Rohr 2 und Rohr 3. Welcher Zusammenhang zu erwarten ist, steht in Abschnitt 4.2.1 (Serien- und Parallelschaltung von Strömungswiderständen). Bestimmen Sie die Unsicherheiten der verglichenen Größen mit Gaußscher Fehlerfortpflanzung.

Aufgabe 6 fragt außerdem: Stimmt der Zusammenhang mit Ihren Erwartungen überein? Warum wurde die Parallelschaltung von Rohr 1 und Rohr 2 nicht als Beispiel für die Messung gewählt?

In [ ]:
# Ihr Code hier


## 5. Turbulenzrohr

Im Turbulenzrohr (Rohr 4) reißt die Strömung hinter einem kleinen Loch ab, es können Verwirbelungen entstehen. Aufgabe 7 verlangt ein eigenes Diagramm von $j$ gegen $\Delta p$. Bei niedrigen Drücken kann von laminarer Strömung ausgegangen werden, deshalb wird zur Orientierung eine Gerade eingezeichnet, die sich nur an diesen Messwerten orientiert.

### Messwerte definieren und Volumenstromstärke berechnen

Hier werden die Sinkstrecken, Druckdifferenzen und Sinkzeiten am Turbulenzrohr definiert (Tabelle 4.6). Berechnen Sie $j$ und $u_j$ wie in Teil 3.

In [ ]:
# Ihr Code hier


### Diagramm der Messwerte erstellen

Es wird $j$ gegen $\Delta p$ mit Fehlerbalken dargestellt. Achten Sie auf den Kurvenverlauf im Vergleich zu den vorherigen Messungen (Aufgabe 7).

In [ ]:
# Ihr Code hier


### Aufgabe 7: Gerade für niedrige Drücke einzeichnen

Legen Sie eine Gerade nur durch die Messwerte bei niedrigen Drücken (Fit mit `odr_fit` wie in Teil 2) und zeichnen Sie sie über den ganzen Druckbereich ein. Welche Messwerte dazugehören, entscheiden Sie am Diagramm. Mit Slicing wie `x[0:n]` werden die ersten `n` Werte eines Arrays ausgewählt.

Zur Erinnerung (ODR), Vorgehen wie beim ersten Fit:

- Modellfunktion `linear_odr(x, beta)`: zuerst `x`, danach `beta`
- Gewichte `weight_x=1/u_x**2` und `weight_y=1/u_y**2`
- Startwerte `beta0` in der richtigen Größenordnung
- Unsicherheiten der Parameter aus `np.sqrt(output.cov_beta[0, 0])` und `np.sqrt(output.cov_beta[1, 1])`

Aufgabe 7 fragt: Gibt es Abweichungen zu dieser Geraden bei höheren Drücken?

In [ ]:
# Ihr Code hier


## 6. Zusammenfassung der Ergebnisse

Zum Schluss werden die wichtigsten Ergebnisse des gesamten Versuchs gesammelt ausgegeben.

Die Zusammenfassung enthält:

1. den Leitwert von Rohr 2 mit der Unsicherheit aus Aufgabe 1,

$$
G \pm u_G
$$

2. Leitwerte und Strömungswiderstände aller vier Messreihen sowie die theoretischen Strömungswiderstände der Rohre 1, 2 und 3,

$$
R \pm u_R
$$

3. die Vergleiche der Rohre aus den Aufgaben 4 bis 6,
4. die Beobachtung am Turbulenzrohr (Aufgabe 7).

Achten Sie darauf, alle Ergebnisse mit passenden Einheiten auszugeben. Denken Sie daran, die Ergebnisse im Bericht mit der richtigen Anzahl signifikanter Stellen anzugeben.

In [ ]:
# Ihr Code hier
